# Chapter 4: Multiple Regression Analysis — Inference

Topics: t-test, confidence intervals, F-tests for linear restrictions.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import scipy.stats as stats
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 4.1 The t-Test

## Step 2 – t-Tests for Individual Coefficients

Under MLR.1–MLR.6 (Gauss-Markov + normality), the t-statistic:

$$t_{\hat{\beta}_j} = \frac{\hat{\beta}_j - \beta_j^0}{\text{se}(\hat{\beta}_j)} \sim t_{n-k-1}$$

**Two-sided test** H₀: $\beta_j = 0$ vs. H₁: $\beta_j \neq 0$:
- Reject H₀ if $|t_{\hat{\beta}_j}| > t_{\alpha/2,\, n-k-1}$
- Equivalently, reject if p-value $< \alpha$

**Standard significance levels:**
| Level | Symbol |
|-------|--------|
| 10% | `*` |
| 5% | `**` |
| 1% | `***` |

In [ ]:
df = woo.dataWoo('wage1')
model = smf.ols('wage ~ educ + exper + tenure + female', data=df).fit()

print("Coefficients, t-stats, and p-values:")
summary_df = pd.DataFrame({
    'coef': model.params,
    't-stat': model.tvalues,
    'p-value': model.pvalues,
    'sig': ['***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
            for p in model.pvalues]
})
print(summary_df.round(4))

## 4.2 Confidence Intervals

## Step 3 – Confidence Intervals for Coefficients

A **95% confidence interval** for $\beta_j$:
$$\hat{\beta}_j \pm t_{0.025,\, n-k-1} \cdot \text{se}(\hat{\beta}_j)$$

Interpretation: if we repeated the sampling procedure many times, 95% of the constructed intervals would contain the true $\beta_j$.

In `statsmodels`: `results.conf_int(alpha=0.05)` returns the lower and upper bounds.

In [ ]:
ci = model.conf_int(alpha=0.05)
ci.columns = ['2.5%', '97.5%']
ci['coef'] = model.params
print("95% Confidence Intervals:")
print(ci[['coef','2.5%','97.5%']].round(4))

# Plot CIs
fig, ax = plt.subplots(figsize=(8, 5))
params = model.params.drop('Intercept')
lower  = ci['2.5%'].drop('Intercept')
upper  = ci['97.5%'].drop('Intercept')
ax.barh(params.index, params, xerr=[params - lower, upper - params],
        color='steelblue', alpha=0.7, capsize=5)
ax.axvline(0, color='red', linestyle='--')
ax.set_title('Coefficient Estimates with 95% CIs')
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_confidence_intervals.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_04_Multiple_Regression_Inference_01.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4.3 Linear Restrictions: F-Tests

## Step 4 – F-Tests for Linear Restrictions

To test a set of $q$ linear restrictions jointly, compare the **restricted** vs. **unrestricted** models:

$$F = \frac{(SSR_r - SSR_{ur})/q}{SSR_{ur}/(n-k-1)} \sim F_{q,\, n-k-1}$$

Equivalently using R²:
$$F = \frac{(R^2_{ur} - R^2_r)/q}{(1-R^2_{ur})/(n-k-1)}$$

**Overall model significance** (H₀: $\beta_1=\cdots=\beta_k=0$, so $q=k$):
$$F = \frac{R^2/k}{(1-R^2)/(n-k-1)}$$

Reject H₀ if F > $F_{\alpha,q,n-k-1}$ or p-value $< \alpha$.

In [ ]:
# Test: H0: β_exper = 0 AND β_tenure = 0
f_test = model.f_test("exper = 0, tenure = 0")
print(f"F-statistic: {f_test.fvalue[0][0]:.4f}")
print(f"p-value:     {f_test.pvalue:.4f}")

# Overall F-test (all slopes = 0)
print(f"\nOverall F-stat: {model.fvalue:.4f}")
print(f"Overall F p-value: {model.f_pvalue:.4f}")